# ⚡ FaceKey Animation Studio — Lightning.ai Cloud Data Collector
### Free 24/7 Cloud Execution (4 CPU Cores, 16 GB RAM)
- **Queue**: `youtube_urls_lightning.txt` (117 Dedicated Videos — Zero Overlap with Colab or Kaggle)
- **Throughput**: 3 Parallel Workers running at 150+ FPS in Turbo Mode
- **Cloud-to-Cloud Sync**: Direct backup via HuggingFace or Service Account to your 5 TB Google Drive (0 MB local data).

In [ ]:
# ==============================================================================
# ⚡ 1-CLICK INSTANT RUNNER: LIGHTNING.AI WORKER
# ==============================================================================
import os, sys, shutil

# 1. Working directories
STUDIO_DIR = "/teamspace/studios/this_studio"
DRIVE_DIR = os.path.join(STUDIO_DIR, "FaceKeyDataset") if os.path.exists(STUDIO_DIR) else "./FaceKeyDataset"
for folder in ("chunks", "checkpoints", "locks", "data"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)

# 2. Update to latest repository code (Self-Healing)
APP_DIR = os.path.join(STUDIO_DIR, "FaceKeyAnimation") if os.path.exists(STUDIO_DIR) else "./FaceKeyAnimation"
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    print("[*] Updating FaceKeyAnimation to latest GitHub main...")
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print("[*] Fresh clone of FaceKeyAnimation from GitHub...")
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Fast Cloud Dependencies & Deno JS solver
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Dedicated Queue for Lightning.ai (Zero Overlap with Colab or Kaggle)
urls_file = os.path.join(APP_DIR, "sessions", "youtube_urls_lightning.txt")

# 5. Launch Collector (3 Workers across 4 CPU cores, 1 Face Only, Turbo Mode)
print("\n[*] Launching FaceKey Turbo Multi-Worker Collector on Lightning.ai (lightning-worker-1)...")
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "lightning-worker-1" \
    --urls-file "{urls_file}" \
    --chunk-size 5 \
    --quality 480p \
    --num-workers 3


---
# 📦 LIGHTNING.AI STORAGE MANAGEMENT & CLOUD UPLOAD (0 MB LOCAL DATA)
> [!TIP]
> Run these cells to check storage and push chunks to your 5 TB Google Drive cloud-to-cloud.

In [ ]:
# 📊 Check Storage & Pending Chunks
!python scripts/cloud_drive_uploader.py --method status --chunks-dir "{DRIVE_DIR}/chunks"


In [ ]:
# 🚀 Push Chunks to Private HuggingFace Dataset (0 MB Local Internet)
HF_TOKEN = "YOUR_HF_WRITE_TOKEN"
HF_REPO = "YOUR_USERNAME/facekey-chunks"

!python scripts/cloud_drive_uploader.py \
    --method hf \
    --chunks-dir "{DRIVE_DIR}/chunks" \
    --hf-token "{HF_TOKEN}" \
    --hf-repo "{HF_REPO}"


In [ ]:
# ☁️ Direct Cloud-to-Cloud Upload to your 5 TB Google Drive (via Service Account)
SERVICE_ACCOUNT_JSON = "/teamspace/studios/this_studio/service_account.json"
GOOGLE_DRIVE_FOLDER_ID = "11VbtMpmNATsrBZxkPLpA2gPTtgaRFNlA"

!python scripts/cloud_drive_uploader.py \
    --method gdrive \
    --chunks-dir "{DRIVE_DIR}/chunks" \
    --service-account "{SERVICE_ACCOUNT_JSON}" \
    --folder-id "{GOOGLE_DRIVE_FOLDER_ID}"
